In [1]:
import jax.numpy as jnp
import roughpy_jax as rpj
import jax
from roughpy_jax.streams import LieIncrementStream
from utils import generate_sinusoidal_timeseries, uniform_intervals, to_list_format, make_incremental
from solver import RoughKernel

ImportError: DLL load failed while importing _jax: An Application Control policy has blocked this file.

In [ ]:
rk = RoughKernel(5, 5)
intervals = uniform_intervals(5)

In [ ]:
B = 100
N = 50
W = 2
key = jax.random.PRNGKey(0)

times, data = generate_sinusoidal_timeseries(key, B, N, W)
_, data_inc = make_incremental(times, data)
times, data = to_list_format(times[:, 1:], data_inc)

In [ ]:
len(times[0]) # N - 1
len(times) # B
data[0].shape # (N-1, W)
len(data) # B

AttributeError: 'list' object has no attribute 'shape'

In [ ]:
Gram = rk.solve_PDE(intervals=intervals, X=data, Y=data, is_Lie=False, times_X = times, times_Y = times)

In [ ]:
# pairwise comparison
Lie_Basis = rpj.LieBasis(depth = 5, width = W)
Tensor_Basis = rpj.to_tensor_basis(Lie_Basis)

X_Lie = LieIncrementStream.from_increments(timestamps=jnp.array(times[0]),
                                           data=data_inc[0],
                                           resolution=5,
                                           input_data_basis=None,
                                           lie_basis=Lie_Basis)

Y_Lie = LieIncrementStream.from_increments(timestamps=jnp.array(times[0]),
                                           data=data_inc[0],
                                           resolution=6,
                                           input_data_basis=None,
                                           lie_basis=Lie_Basis)

In [6]:
X_Lie.signature().basis

TensorBasis(2, 5)

In [7]:
Gram[0, 0], rpj.tensor_pairing(X_Lie.signature(), Y_Lie.signature())

(Array(16.247982, dtype=float32), Array([18.488033], dtype=float32))